# handoff_v1.parquet 분석 + transition-models 호환성 검토

팀원([송동선])이 배포한 통합 데이터셋 v1의 구조를 분석하고
본인의 기존 77/87-dim feature 체계와의 호환성을 검증한다.

## 목적
1. handoff_v1 컬럼 구조 및 새 features 파악
2. 본인 77/87-dim feature와 overlap 확인
3. 통합 시나리오(A/B/C) 평가 → Phase 9 방향 결정

## 참고
- parquet 미존재 시: 스키마 JSON(`handoff_v1_schema.json`) 기반 분석
- parquet 존재 시: 섹션 5에서 실제 통계 추가

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

REPO_ROOT = Path("..").resolve()
TEAMMATE_REPO = REPO_ROOT.parent / "data"
SCHEMA_PATH = TEAMMATE_REPO / "data" / "outputs" / "handoff_v1_schema.json"
PARQUET_PATH = TEAMMATE_REPO / "data" / "outputs" / "handoff_v1.parquet"

print(f"스키마 경로: {SCHEMA_PATH}")
print(f"parquet 경로: {PARQUET_PATH}")
print(f"parquet 존재: {'Yes' if PARQUET_PATH.exists() else 'No (스키마 기반 분석)'}")

## 1. handoff_v1 기본 정보

스키마 JSON에서 컬럼 구조, 행 수, 시즌 범위 파악.

In [ ]:
with open(SCHEMA_PATH, encoding="utf-8") as f:
    schema = json.load(f)

n_rows = schema["n_rows"]
n_cols = schema["n_columns"]
row_counts = schema["row_counts"]

print(f"버전      : {schema['version']}")
print(f"행 수     : {n_rows:,}")
print(f"컬럼 수   : {n_cols}")
print(f"join keys : {schema['join_keys']}")
print(f"시즌      : 2022-2025 (statcast_*_clean.parquet 2022..2025)")
print()
print("Row counts:")
for k, v in row_counts.items():
    pct = v / n_rows * 100
    print(f"  {k:<30}: {v:>10,}  ({pct:.4f}%)")

In [ ]:
# 컬럼 소스별 분류
cols = schema["columns"]

UMAP_COLS = [c for c in cols if c.startswith("umap_")]
ARSENAL_FUNC = [c for c in cols if c.startswith("arsenal_func_")]
ARSENAL_MOMENT = [c for c in cols if c.startswith("arsenal_moment_")]
COUNT_CLUSTER = [c for c in cols if "count_cluster" in c]
META = ["arsenal_fallback", "batter_cluster_ready"]
STATCAST = [c for c in cols if not any(c.startswith(p) for p in ["umap_", "arsenal_", "count_cluster", "batter_"])]

print(f"컬럼 그룹별 수:")
print(f"  Statcast 원본  : {len(STATCAST):>3}개")
print(f"  UMAP 임베딩    : {len(UMAP_COLS):>3}개  {UMAP_COLS}")
print(f"  카운트 클러스터: {len(COUNT_CLUSTER):>3}개  {COUNT_CLUSTER}")
print(f"  Arsenal func   : {len(ARSENAL_FUNC):>3}개  (arsenal_func_00..31)")
print(f"  Arsenal moment : {len(ARSENAL_MOMENT):>3}개  (arsenal_moment_00..19)")
print(f"  메타           : {len(META):>3}개  {META}")
print(f"  합계           : {len(STATCAST)+len(UMAP_COLS)+len(COUNT_CLUSTER)+len(ARSENAL_FUNC)+len(ARSENAL_MOMENT)+len(META):>3}개")

## 2. 새 Features 자세한 분석

v1에서 추가된 3가지 feature 그룹:
- **UMAP 임베딩**: 구종 특성 공간의 비선형 압축 (5d 모델용 + 2d 시각화용)
- **카운트 클러스터**: 볼카운트 상황을 4-way Ward 클러스터로 분류
- **투수 레퍼토리 벡터**: 투수별 구종 특성 (func 32d + moment 20d = 52d)

In [ ]:
# 카운트 클러스터 정의 (스키마에서)
cc_desc = cols["count_cluster_id"]["description"]
print("카운트 클러스터 정의:")
print(f"  {cc_desc}")
print()

# 이론적 카운트 → 클러스터 매핑 시각화
# 볼카운트 (0-3) × 스트라이크 (0-2) = 12개 조합
count_cluster_map = {
    "0-0": 1, "0-1": 1, "0-2": 2,
    "1-0": 1, "1-1": 1, "1-2": 2,
    "2-0": 1, "2-1": 1, "2-2": 2,
    "3-0": 3, "3-1": 0, "3-2": 0,
}
cluster_labels = {0: "pitcher-favored", 1: "0strikes", 2: "hitter-favored", 3: "3-0 take"}
cluster_colors = {0: "#e74c3c", 1: "#3498db", 2: "#27ae60", 3: "#9b59b6"}

fig, ax = plt.subplots(figsize=(8, 4))
for count, cluster in count_cluster_map.items():
    b, s = map(int, count.split("-"))
    ax.scatter(b, s, c=cluster_colors[cluster], s=500, zorder=3)
    ax.text(b, s + 0.08, cluster_labels[cluster], ha="center", fontsize=8)
ax.set_xlabel("Balls")
ax.set_ylabel("Strikes")
ax.set_title("Count Cluster 매핑 (4-way Ward, count_cluster_id)")
ax.set_xticks([0, 1, 2, 3])
ax.set_yticks([0, 1, 2])
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.3, 2.5)
ax.grid(True, alpha=0.3)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=f"{k}: {cluster_labels[k]}") for k, v in cluster_colors.items()]
ax.legend(handles=legend_elements, loc="upper right", fontsize=9)
plt.tight_layout()

In [ ]:
# 새 features 차원 요약 시각화
feature_groups = {
    "UMAP 5d\n(구종 임베딩)": 5,
    "UMAP 2d\n(시각화용)": 2,
    "Count cluster\n(4-way)": 1,
    "Arsenal func\n(32d)": 32,
    "Arsenal moment\n(20d)": 20,
}
colors = ["#3498db", "#95a5a6", "#e67e22", "#2ecc71", "#27ae60"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(list(feature_groups.keys()), list(feature_groups.values()), color=colors, alpha=0.85, edgecolor="white")
for bar, val in zip(bars, feature_groups.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), ha="center", fontsize=12, fontweight="bold")
ax.set_ylabel("차원 수")
ax.set_title("handoff_v1 새 Feature 그룹별 차원 수 (모델 입력 가능: UMAP5d + count1 + arsenal52 = 58d)")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylim(0, 40)
total_usable = 5 + 1 + 32 + 20
ax.text(0.98, 0.95, f"모델 입력 가능: {total_usable}d\n(UMAP2d 제외)",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
plt.tight_layout()

In [ ]:
# UMAP/arsenal 품질 지표 (스키마 기반)
embed_miss = row_counts["embed_miss"]
fallback = row_counts["arsenal_fallback_rows"]
unmapped = row_counts["count_cluster_unmapped"]

print("새 feature 품질 지표:")
print(f"  UMAP embed_miss    : {embed_miss:>6,}건  ({embed_miss/n_rows*100:.4f}%)  → NaN imputation 또는 drop")
print(f"  Arsenal fallback   : {fallback:>6,}건  ({fallback/n_rows*100:.2f}%)  → fleet-mean 대체")
print(f"  Count cluster -1   : {unmapped:>6,}건  ({unmapped/n_rows*100:.4f}%)  → 완전 매핑")
print()
print("UMAP 컬럼 설명:")
for c in UMAP_COLS:
    print(f"  {c}: {cols[c]['description']}")

## 3. 본인 데이터와의 호환성

전처리 기준 컬럼(`CONTINUOUS_FEATURES`, `CATEGORICAL_FEATURES`, 등)이
handoff_v1에 모두 존재하는지 확인.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / "src"))
from data.features import CONTINUOUS_FEATURES, PITCH_TYPES, ZONES

v1_cols = set(cols.keys())

# 77-dim Model B 원본 컬럼
MODEL_B_SOURCE = set(CONTINUOUS_FEATURES) | {
    "pitch_type", "zone", "balls", "strikes", "outs_when_up",
    "on_1b", "on_2b", "on_3b", "stand", "p_throws", "inning",
}
# 87-dim Model C 원본 컬럼 (inning 대신 outcome)
MODEL_C_SOURCE = set(CONTINUOUS_FEATURES) | {
    "pitch_type", "zone", "balls", "strikes", "outs_when_up",
    "on_1b", "on_2b", "on_3b", "stand", "p_throws",
    "description", "events", "hit_location",
}

missing_b = MODEL_B_SOURCE - v1_cols
missing_c = MODEL_C_SOURCE - v1_cols
join_keys = ["game_pk", "at_bat_number", "pitch_number"]

print("=" * 50)
print(f"77-dim Model B 원본 ({len(MODEL_B_SOURCE)}개 컬럼 필요)")
if missing_b:
    print(f"  !! 누락: {sorted(missing_b)}")
else:
    print(f"  ✓ 전부 v1에 존재 → 77-dim 100% 재추출 가능")

print()
print(f"87-dim Model C 원본 ({len(MODEL_C_SOURCE)}개 컬럼 필요)")
if missing_c:
    print(f"  !! 누락: {sorted(missing_c)}")
else:
    print(f"  ✓ 전부 v1에 존재 → 87-dim 100% 재추출 가능")

print()
print("JOIN key 컬럼 존재 여부:")
for k in join_keys:
    mark = "✓" if k in v1_cols else "✗"
    print(f"  {mark} {k}")

extra_dim = 5 + 1 + 32 + 20  # umap5d + count_cluster + arsenal_func + arsenal_moment
print()
print(f"v1 추가 가능 차원: {extra_dim}d")
print(f"  77-dim → {77 + extra_dim}-dim 확장")
print(f"  87-dim → {87 + extra_dim}-dim 확장")

In [ ]:
# 시즌 비교 시각화
tm_3s_rows = 749880 + 385320 + 353776   # transition-models 3시즌 processed
tm_2s_rows = 1488976 - 749880           # 대략적 2시즌

# v1 시즌별 추정 (4시즌 균등 가정)
v1_per_season = n_rows / 4

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: 총 행 수 비교
labels = ["v1 전체\n(2022-2025)", "v1 2022-2024\n(추정)", "본인 processed\n(2022-2024)"]
values = [n_rows, v1_per_season * 3, tm_3s_rows]
colors_bar = ["#3498db", "#85c1e9", "#e74c3c"]
bars = axes[0].bar(labels, values, color=colors_bar, alpha=0.85, edgecolor="white")
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20000,
                 f"{val:,.0f}", ha="center", fontsize=9)
axes[0].set_ylabel("행 수")
axes[0].set_title("데이터셋 행 수 비교")
axes[0].set_ylim(0, n_rows * 1.15)

# 오른쪽: 데이터 구조 비교 (차원)
dim_labels = ["본인 Model B\n(77-dim)", "본인 Model C\n(87-dim)", "v1+Model B\n(135-dim)", "v1+Model C\n(145-dim)"]
dim_values = [77, 87, 77 + extra_dim, 87 + extra_dim]
base_values = [0, 0, 77, 87]
extra_values = [0, 0, extra_dim, extra_dim]
colors_dim = ["#e74c3c", "#e74c3c", "#e74c3c", "#e74c3c"]
colors_extra = ["#27ae60", "#27ae60", "#27ae60", "#27ae60"]
axes[1].bar(dim_labels[:2], dim_values[:2], color="#e74c3c", alpha=0.85, edgecolor="white", label="기존 dim")
axes[1].bar(dim_labels[2:], base_values[2:], color="#e74c3c", alpha=0.85, edgecolor="white")
axes[1].bar(dim_labels[2:], extra_values[2:], bottom=base_values[2:], color="#27ae60", alpha=0.85, edgecolor="white", label=f"v1 추가 ({extra_dim}d)")
for i, (label, val) in enumerate(zip(dim_labels, dim_values)):
    axes[1].text(i, val + 1, str(val), ha="center", fontsize=10, fontweight="bold")
axes[1].set_ylabel("차원 수")
axes[1].set_title("입력 차원 비교")
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 170)

plt.tight_layout()

## 4. parquet 실제 통계 (parquet 수령 후 실행)

아래 셀은 `handoff_v1.parquet` 파일이 있을 때만 실행.

필요 파일 (팀원에게 요청 또는 직접 빌드):
- `C:\Users\zpfh1\Projects\data\data\clean\statcast_{year}_clean.parquet` (2022-2025)
- `C:\Users\zpfh1\Projects\clustering\outputs\pitch_embeddings.parquet`
- `C:\Users\zpfh1\Projects\clustering\outputs\count_clusters_v1.parquet`
- `C:\Users\zpfh1\Projects\clustering\outputs\arsenal_features_v1.parquet`

In [ ]:
if not PARQUET_PATH.exists():
    print("[SKIP] handoff_v1.parquet 미존재. 팀원에게 파일 요청 또는 assemble_v1.py 빌드 필요.")
    print("  → 스키마 기반 분석으로 충분히 호환성 확인 완료 (위 섹션 1-3 참고)")
else:
    df_v1 = pd.read_parquet(PARQUET_PATH)
    print(f"로드 완료: {df_v1.shape}")

    # 시즌 분포
    print("\n시즌 분포:")
    print(df_v1["game_year"].value_counts().sort_index())

    # UMAP null 비율
    umap_5d = [f"umap_5d_{i}" for i in range(5)]
    print("\nUMAP null 비율:")
    for c in umap_5d:
        print(f"  {c}: {df_v1[c].isna().mean()*100:.4f}%")

    # count_cluster 분포
    print("\ncount_cluster_id 분포:")
    print(df_v1["count_cluster_id"].value_counts().sort_index())

    # overlap 분석
    raw_dir = REPO_ROOT / "data" / "raw"
    raw_parts = []
    for yr in (2022, 2023, 2024):
        p = raw_dir / str(yr) / f"statcast_{yr}.parquet"
        if p.exists():
            raw_parts.append(pd.read_parquet(p, columns=["game_pk", "at_bat_number", "pitch_number"]))
    if raw_parts:
        raw = pd.concat(raw_parts, ignore_index=True).dropna()
        v1_2024 = df_v1[df_v1["game_year"].isin([2022, 2023, 2024])][["game_pk", "at_bat_number", "pitch_number"]].dropna()
        merged = v1_2024.merge(raw, on=["game_pk", "at_bat_number", "pitch_number"], how="inner")
        print(f"\n=== Overlap (2022-2024) ===")
        print(f"v1 rows (2022-2024): {len(v1_2024):,}")
        print(f"본인 raw rows      : {len(raw):,}")
        print(f"inner join         : {len(merged):,}")
        print(f"overlap rate       : {len(merged)/len(raw)*100:.2f}%")

## 5. 통합 시나리오 평가 및 결론

스키마 분석 결과를 바탕으로 3가지 시나리오 평가.

In [ ]:
# 시나리오 요약 표
scenario_data = {
    "시나리오": ["A: feature 추가 재학습", "B: 병행 비교 (권장)", "C: v1 미사용"],
    "입력 dim": ["77→135 또는 87→145", "기존 77/87 + v1 별도", "기존 77/87 유지"],
    "재학습 필요": ["전체 (Model C ~10h)", "v1-enhanced 트랙만", "없음"],
    "발표 narrative": ["통합 데이터 강화", "기존 vs 통합 비교", "5모델 비교 (기존)"],
    "추가 작업": ["중~대", "중", "없음"],
}
df_scenario = pd.DataFrame(scenario_data)
print(df_scenario.to_string(index=False))

In [ ]:
# 최종 분석 결론 요약
summary = {
    "기본 정보": {
        "rows": f"{n_rows:,}",
        "cols": n_cols,
        "시즌": "2022-2025 (4시즌)",
    },
    "새 features": {
        "UMAP 5d": "구종 특성 공간 비선형 압축 (embed_miss 0.07%)",
        "count_cluster_id": "4-way Ward 클러스터 (완전 매핑)",
        "arsenal 52d": f"투수 레퍼토리 (fallback {row_counts['arsenal_fallback_rows']/n_rows*100:.2f}%)",
    },
    "호환성": {
        "77-dim Model B": "100% 재추출 가능",
        "87-dim Model C": "100% 재추출 가능",
        "JOIN key": "game_pk + at_bat_number + pitch_number 모두 존재",
    },
    "v1 제약": {
        "batter_cluster": "없음 (batter_cluster_ready=False, v2 예정)",
        "시즌 초과": "v1은 2025 포함, 본인은 2024까지",
        "embed_miss": "2,171건 NaN (0.07%) → 처리 전략 필요",
        "scaler": "본인 StandardScaler 재fit 필요 (v1 기준)",
    },
    "권장 시나리오": "B — Phase 9 병행 + v1-enhanced 별도 트랙",
}

for section_name, items in summary.items():
    print(f"\n{'='*50}")
    print(f"[{section_name}]")
    if isinstance(items, dict):
        for k, v in items.items():
            print(f"  {k:<20}: {v}")
    else:
        print(f"  {items}")

## 결론

### 호환성 평가
- **100% 호환**: 77-dim(Model B)·87-dim(Model C) 원본 컬럼이 v1에 모두 존재
- **같은 pitch 행**: JOIN key(`game_pk, at_bat_number, pitch_number`) 존재 → 기존 라벨 재사용 가능
- **58차원 추가 가능**: UMAP(5) + count_cluster(1) + arsenal(52)

### v1 제약
- `batter_cluster` 없음 → 타자 특성 반영 불가 (v2 대기)
- 2,171건 UMAP NaN → drop 또는 imputation 필요
- `StandardScaler`는 v1 기준으로 재fit 필요
- 2025 시즌 추가 → 본인 3시즌과 시즌 범위 불일치

### 권장: **시나리오 B**
- Phase 9(5모델 비교)를 기존 2022-2024 데이터로 계속 진행
- v1-enhanced(77+58=135-dim) 실험을 별도 트랙으로
- 두 결과 비교 → "feature engineering 효과" 정량화
- **단, parquet 직접 수령(팀원 요청) 후 overlap JOIN 확인 필수**